In [2]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

# Load environment
load_dotenv()

# Load all text files from directory
loader = DirectoryLoader('company_docs/', glob='*.txt', loader_cls=TextLoader)
documents = loader.load()

print(f'Loaded {len(documents)} documents')
print(f'First doc preview: {documents[0].page_content[:200]}...')

Loaded 3 documents
First doc preview: Employee Benefits Overview

Health Insurance: The company provides full health insurance coverage for employees, with 50% coverage extended to dependents. Coverage begins on the first day of employmen...


In [3]:
# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # Characters per chunk
    chunk_overlap=50,      # Overlap between chunks
    length_function=len,
    separators=['\n\n', '\n', '. ', ' ', '']
)

# Split documents
chunks = text_splitter.split_documents(documents)
print(f'Split into {len(chunks)} chunks')

print('\nSample chunks:')
for i, chunk in enumerate(chunks[:3]):
    print(f'\nChunk {i+1}:')
    print(chunk.page_content)
    print(f'Length: {len(chunk.page_content)} chars')

Split into 3 chunks

Sample chunks:

Chunk 1:
Employee Benefits Overview

Health Insurance: The company provides full health insurance coverage for employees, with 50% coverage extended to dependents. Coverage begins on the first day of employment.

Retirement Plan: Employees are eligible for a retirement savings plan with a 5% company match after 6 months of employment.

Wellness Program: Employees receive an annual wellness stipend of $300 for gym memberships, fitness classes, or wellness apps.
Length: 455 chars

Chunk 2:
Employee Handbook - HR Policies

Vacation Policy: All full-time employees receive 15 days of paid vacation per year. Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week. Remote work requires manager approval.

Parental Leave: 12 weeks paid parental leave for primary caregivers. 6 weeks paid leave for secondary caregivers.
Length: 402 chars

Chunk 3:
IT and Equipment Policy

Laptop Policy:

In [4]:
def simple_search(query, chunks, top_k=3):
    """
    Simple keyword-based search
    Returns top_k most relevant chunks
    """
    query_lower = query.lower()
    
    # Score each chunk
    scored_chunks = []
    for chunk in chunks:
        content_lower = chunk.page_content.lower()
        # Count keyword matches
        score = 0
        for word in query_lower.split():
            score += content_lower.count(word)
        if score > 0:
            scored_chunks.append((score, chunk))
    
    # Sort by score and return top k
    scored_chunks.sort(reverse=True, key=lambda x: x[0])
    return [chunk for score, chunk in scored_chunks[:top_k]]

# Test it
query = 'What is the vacation policy?'
relevant = simple_search(query, chunks)
print(f'Found {len(relevant)} relevant chunks:')
for i, chunk in enumerate(relevant):
    print(f'\n--- Chunk {i+1} ---')
    print(chunk.page_content)

Found 3 relevant chunks:

--- Chunk 1 ---
Employee Handbook - HR Policies

Vacation Policy: All full-time employees receive 15 days of paid vacation per year. Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week. Remote work requires manager approval.

Parental Leave: 12 weeks paid parental leave for primary caregivers. 6 weeks paid leave for secondary caregivers.

--- Chunk 2 ---
Employee Benefits Overview

Health Insurance: The company provides full health insurance coverage for employees, with 50% coverage extended to dependents. Coverage begins on the first day of employment.

Retirement Plan: Employees are eligible for a retirement savings plan with a 5% company match after 6 months of employment.

Wellness Program: Employees receive an annual wellness stipend of $300 for gym memberships, fitness classes, or wellness apps.

--- Chunk 3 ---
IT and Equipment Policy

Laptop Policy: All full-time employees a

In [5]:
# Test multiple queries
test_queries = [
    'How many vacation days do employees get?',
    'What is the remote work policy?',
    'Tell me about parental leave',
]

for query in test_queries:
    print(f'\nQuery: {query}')
    results = simple_search(query, chunks, top_k=2)
    print(f'Found {len(results)} relevant chunks')
    if results:
        print(f'Top result: {results[0].page_content[:100]}...')


Query: How many vacation days do employees get?
Found 2 relevant chunks
Top result: Employee Handbook - HR Policies

Vacation Policy: All full-time employees receive 15 days of paid va...

Query: What is the remote work policy?
Found 2 relevant chunks
Top result: Employee Handbook - HR Policies

Vacation Policy: All full-time employees receive 15 days of paid va...

Query: Tell me about parental leave
Found 2 relevant chunks
Top result: Employee Handbook - HR Policies

Vacation Policy: All full-time employees receive 15 days of paid va...


In [7]:
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

def rag_query(query, chunks, top_k=3):
    """
    RAG pipeline: Retrieve → Generate
    """
    # Step 1: Retrieve relevant chunks
    relevant_chunks = simple_search(query, chunks, top_k)
    if not relevant_chunks:
        return 'No relevant information found in documents.'
    
    # Step 2: Build context
    context = '\n\n---\n\n'.join([chunk.page_content for chunk in relevant_chunks])
    
    # Step 3: Create prompt
    prompt = f'''You are a helpful assistant. Answer the question using ONLY the context provided below. If the answer is not in the context, say so.

Context:
{context}

Question: {query}

Answer:'''
    
    # Step 4: Generate answer
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt
    )
    return response.text

In [8]:
# Test questions
questions = [
    'How many vacation days do full-time employees get?',
    'Can employees work from home?',
    'What is the parental leave policy?',
    'What is the dress code?'  # Not in docs
]

for question in questions:
    print(f'\n{"="*60}')
    print(f'Q: {question}')
    print(f'{"="*60}')
    answer = rag_query(question, chunks)
    print(f'A: {answer}')


Q: How many vacation days do full-time employees get?
A: Full-time employees receive 15 days of paid vacation per year.

Q: Can employees work from home?
A: Yes. According to the context, employees may work remotely up to 3 days per week, provided they have manager approval. Remote employees must also connect via company VPN when accessing internal systems from outside the office.

Q: What is the parental leave policy?
A: Based on the provided context, the parental leave policy offers:
- 12 weeks of paid parental leave for primary caregivers.
- 6 weeks of paid leave for secondary caregivers.

Q: What is the dress code?
A: The provided context does not contain information about the dress code.


In [9]:
def ask_without_rag(question):
    """
    Ask Gemini directly (no context)
    """
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=f'You are a helpful HR assistant. {question}'
    )
    return response.text

# Compare
question = 'How many vacation days do employees get?'
print('WITHOUT RAG:')
print(ask_without_rag(question))
print('\nWITH RAG:')
print(rag_query(question, chunks))

WITHOUT RAG:
To give you the exact number, I would need to know which company’s policy you are asking about! 

As an AI, I don't have access to your specific company's employee handbook or HR portal unless you share those details with me. 

However, if you are looking for general standard practices:

* **United States:** Standard practice for full-time private-sector employees typically starts at **10 to 15 paid vacation days** (PTO) per year, often increasing with years of tenure.
* **European Union / UK:** Most countries legally mandate a minimum of **20 to 28 paid vacation days** per year.
* **Unlimited PTO:** Some companies (especially in tech) offer "Unlimited PTO," subject to manager approval.

### How to find your specific allowance:
1. **Share the policy:** If you paste the vacation/PTO section of your company handbook here, I can read it and break down the exact rules for you.
2. **Check internal resources:** Look at your original **Offer Letter**, log into your company’s HR p